# Retina AI — Student Dropout Risk Prediction

**A complete end-to-end machine learning pipeline combining gradient-boosted trees, a multimodal deep learning network, and a tuned blended ensemble to predict student dropout risk (Low, Medium, High) from tabular records, attendance time-series, and counsellor notes.**

---

### What this notebook does

| Stage | Description |
|-------|-------------|
| 1 | Load raw CSVs — training labels, test records, attendance series, counsellor notes |
| 2 | Engineer attendance aggregate and trend features |
| 3 | Parse and encode counsellor notes with keyword risk signals |
| 4 | Build tabular derived features from CGPA and backlog trajectories |
| 5 | Merge all feature tables into a single design matrix |
| 6 | Encode low-cardinality categoricals with label encoding |
| 7 | Apply out-of-fold target encoding for high-cardinality note identifiers |
| 8 | Assemble the final feature list |
| 9 | Train a 4-model GBM ensemble (LightGBM, XGBoost, CatBoost, ExtraTrees) with 5-fold CV |
| 10 | Train a multimodal neural network (Tabular MLP + Attendance BiLSTM + Notes BiLSTM) |
| 11 | Blend GBM and DL predictions, tune class weights, evaluate, and save submission |

---

**Final Result:** Accuracy 75.67% | Macro-F1 0.7120

---
## 0. Imports and Global Configuration

All libraries are imported in one place so every dependency is visible upfront. The random seed is fixed globally across NumPy, Python, and PyTorch to make every run fully reproducible. The device is detected automatically so the same notebook runs on CPU or GPU without any code change.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.ensemble import ExtraTreesClassifier

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED   = 42
TARGET = 'dropout_risk'

np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hardware ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')

---
## 1. Load Raw Data

Four CSV files are loaded from disk.

| File | Contents |
|------|----------|
| `train.csv` | Student records with the target label `dropout_risk` |
| `test.csv` | Student records without labels (to be predicted) |
| `Attendance_series.csv` | Weekly per-subject attendance percentages across semesters |
| `Counsellor_notes.csv` | Free-text counsellor observations per student |

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
att   = pd.read_csv('Attendance_series.csv')
notes = pd.read_csv('Counsellor_notes.csv')

print(f'Train shape  : {train.shape}')
print(f'Test shape   : {test.shape}')
print(f'Attendance   : {att.shape}')
print(f'Notes        : {notes.shape}')

In [ ]:
# Quick look at the target distribution
print(train[TARGET].value_counts())
train.head(3)

---
## 2. Attendance Aggregate Features

The raw attendance series is a long-format table with one row per (student, semester, week, subject). From this we derive five groups of features.

- **Overall statistics** — mean, standard deviation, min, max attendance across the entire record
- **Semester-level means** — average attendance per semester (pivoted wide)
- **Subject-level means** — average attendance per subject (pivoted wide)
- **Linear trend (slope)** — least-squares slope of attendance over the ordered timeline to capture whether a student is improving or declining
- **Low-attendance count** — number of weeks where attendance fell below 50%, a direct risk signal
- **Derived delta features** — difference between semester 3 and semester 1 means, and between semester 3 and semester 2 means

In [ ]:
# ── Create an ordered global time index across all semesters and weeks ────────
att['t'] = (att['semester'] - 1) * 8 + att['week']

In [ ]:
# ── Overall attendance statistics per student ─────────────────────────────────
overall = (
    att.groupby('student_id')['attendance_pct']
    .agg(att_mean='mean', att_std='std', att_min='min', att_max='max')
    .reset_index()
)

In [ ]:
# ── Semester-level mean attendance (wide format) ──────────────────────────────
sem_pivot = att.pivot_table(
    index='student_id', columns='semester',
    values='attendance_pct', aggfunc='mean'
)
sem_pivot.columns = [f'att_sem{c}_mean' for c in sem_pivot.columns]
sem_pivot = sem_pivot.reset_index()

In [ ]:
# ── Subject-level mean attendance (wide format) ───────────────────────────────
subj_pivot = att.pivot_table(
    index='student_id', columns='subject',
    values='attendance_pct', aggfunc='mean'
)
subj_pivot.columns = [f'att_{c}_mean' for c in subj_pivot.columns]
subj_pivot = subj_pivot.reset_index()

In [ ]:
# ── Linear attendance trend via ordinary least squares (vectorized) ───────────
agg = att.groupby('student_id').agg(
    n    = ('t', 'count'),
    sum_t  = ('t', 'sum'),
    sum_y  = ('attendance_pct', 'sum')
)

att['ty'] = att['t'] * att['attendance_pct']
att['tt'] = att['t'] ** 2

agg2 = att.groupby('student_id').agg(
    sum_ty = ('ty', 'sum'),
    sum_tt = ('tt', 'sum')
)
agg = agg.join(agg2)

agg['att_slope'] = (
    (agg['n'] * agg['sum_ty'] - agg['sum_t'] * agg['sum_y'])
    / (agg['n'] * agg['sum_tt'] - agg['sum_t'] ** 2)
)
slope = agg[['att_slope']].reset_index()

In [ ]:
# ── Count of weeks with attendance below 50% ──────────────────────────────────
low_att = (
    att.groupby('student_id')['attendance_pct']
    .apply(lambda x: (x < 0.5).sum())
    .reset_index(name='att_low_weeks')
)

In [ ]:
# ── Merge all attendance feature tables ───────────────────────────────────────
att_feats = (
    overall
    .merge(sem_pivot,  on='student_id')
    .merge(subj_pivot, on='student_id')
    .merge(slope,      on='student_id')
    .merge(low_att,    on='student_id')
)

att_feats['att_recent_vs_early'] = att_feats['att_sem3_mean'] - att_feats['att_sem1_mean']
att_feats['att_sem3_2_diff']     = att_feats['att_sem3_mean'] - att_feats['att_sem2_mean']

print(f'Attendance feature table shape: {att_feats.shape}')

---
## 3. Counsellor Notes Features

Each student has a short free-text note written by a counsellor. Three signal types are extracted.

- **Note identifier** — a label-encoded integer representing the exact note text, later target-encoded
- **Situation identifier** — the first sentence of the note is isolated as a situation descriptor and label-encoded separately
- **Keyword risk flags** — three binary flags indicating whether the note contains high-risk, medium-risk, or positive-outcome keywords

In [ ]:
# ── Label encode the full note text ──────────────────────────────────────────
le_note = LabelEncoder()
notes['note_id'] = le_note.fit_transform(notes['counsellor_note'])

In [ ]:
# ── Extract the situation (first sentence) and encode it ─────────────────────
parts = notes['counsellor_note'].str.rstrip('.').str.split('. ', n=1, expand=True, regex=False)
notes['situation'] = parts[0]

le_sit = LabelEncoder()
notes['situation_id'] = le_sit.fit_transform(notes['situation'])

In [ ]:
# ── Keyword risk flags (regex-based, case-insensitive) ───────────────────────
risk_kws = 'demotivat|financial|health|emergency|severe|multiple backlogs|part-time hours|unresponsive'
warn_kws = 'stress|struggl|focus|attendance|absence|family|monitor'
good_kws = 'well|good|no major issues|no further action|active'

notes['note_high_risk_kw'] = notes['counsellor_note'].str.lower().str.contains(risk_kws).astype(int)
notes['note_med_risk_kw']  = notes['counsellor_note'].str.lower().str.contains(warn_kws).astype(int)
notes['note_low_risk_kw']  = notes['counsellor_note'].str.lower().str.contains(good_kws).astype(int)

print(notes[['note_high_risk_kw','note_med_risk_kw','note_low_risk_kw']].mean().round(3))

---
## 4. Tabular Derived Features

From the base student record we create trajectory and interaction features.

**CGPA features**
- Summary statistics across the four semesters: mean, standard deviation, minimum
- Overall trend (semester 4 minus semester 1)
- Consecutive semester deltas: d21, d32, d43

**Backlog features**
- Total backlogs accumulated across all three semesters
- Overall trend (semester 3 minus semester 1)
- Consecutive semester deltas: d21, d32

**Missingness indicators**
- Binary flags for missing commute time and missing parental education, so the model can learn from the fact of missingness

**Imputation**
- Commute time is filled with the training-set median (computed before touching the test set to prevent leakage)
- Parental education is filled with the string `Unknown`

In [ ]:
# ── CGPA and backlog features applied identically to train and test ───────────
for df in [train, test]:
    cg = ['cgpa_sem1', 'cgpa_sem2', 'cgpa_sem3', 'cgpa_sem4']

    df['cgpa_mean']  = df[cg].mean(axis=1)
    df['cgpa_std']   = df[cg].std(axis=1)
    df['cgpa_min']   = df[cg].min(axis=1)
    df['cgpa_trend'] = df['cgpa_sem4'] - df['cgpa_sem1']
    df['cgpa_d21']   = df['cgpa_sem2'] - df['cgpa_sem1']
    df['cgpa_d32']   = df['cgpa_sem3'] - df['cgpa_sem2']
    df['cgpa_d43']   = df['cgpa_sem4'] - df['cgpa_sem3']

    df['total_backlogs'] = df['backlogs_sem1'] + df['backlogs_sem2'] + df['backlogs_sem3']
    df['backlog_trend']  = df['backlogs_sem3'] - df['backlogs_sem1']
    df['bl_d21']         = df['backlogs_sem2'] - df['backlogs_sem1']
    df['bl_d32']         = df['backlogs_sem3'] - df['backlogs_sem2']

    df['commute_missing']    = df['commute_time_mins'].isnull().astype(int)
    df['parent_edu_missing'] = df['parent_education'].isnull().astype(int)

In [ ]:
# ── Impute missing values (median computed on train only to avoid leakage) ────
commute_median = train['commute_time_mins'].median()

for df in [train, test]:
    df['commute_time_mins'] = df['commute_time_mins'].fillna(commute_median)
    df['parent_education']  = df['parent_education'].fillna('Unknown')

print(f'Commute median used for imputation: {commute_median}')

---
## 5. Merge All Feature Tables

The attendance features, the selected counsellor note columns, and the tabular base data are joined on `student_id` using left joins. After merging, two additional cross-table interaction features are computed.

- `att_d21` and `att_d32`: semester-over-semester attendance deltas (parallel to the CGPA deltas)
- `cgpa_div_backlog`: CGPA per unit of total backlog burden — a measure of academic efficiency under pressure
- `cgpa_x_backlog`: the product of CGPA and backlog count, which is a different interaction capturing combined academic load

In [ ]:
notes_cols = [
    'student_id', 'note_id', 'situation_id',
    'note_high_risk_kw', 'note_med_risk_kw', 'note_low_risk_kw'
]

train = (train
         .merge(att_feats,         on='student_id', how='left')
         .merge(notes[notes_cols], on='student_id', how='left'))

test  = (test
         .merge(att_feats,         on='student_id', how='left')
         .merge(notes[notes_cols], on='student_id', how='left'))

print(f'Train after merge: {train.shape}')
print(f'Test  after merge: {test.shape}')

In [ ]:
# ── Cross-table interaction features ─────────────────────────────────────────
for df in [train, test]:
    df['att_d21'] = df['att_sem2_mean'] - df['att_sem1_mean']
    df['att_d32'] = df['att_sem3_mean'] - df['att_sem2_mean']

    df['cgpa_div_backlog'] = df['cgpa_mean'] / (1 + df['total_backlogs'])
    df['cgpa_x_backlog']   = df['cgpa_mean'] * (1 + df['total_backlogs'])

---
## 6. Encode Low-Cardinality Categoricals

Five low-cardinality categorical columns are label-encoded. The encoder is fitted on the union of train and test values to handle any categories that appear only in the test set.

In [ ]:
cat_cols = ['branch', 'gender', 'hostel_status', 'family_income', 'parent_education']

for c in cat_cols:
    le   = LabelEncoder()
    full = pd.concat([train[c], test[c]], axis=0).astype(str)
    le.fit(full)

    train[c + '_enc'] = le.transform(train[c].astype(str))
    test[c  + '_enc'] = le.transform(test[c].astype(str))

print('Encoded columns:', [c + '_enc' for c in cat_cols])

---
## 7. Out-of-Fold Target Encoding

The `note_id` and `situation_id` columns have high cardinality. A naive mean-target encoding computed on the full training set would leak the label into the features. Instead, we use a 5-fold out-of-fold scheme combined with Bayesian smoothing toward the global class prior.

For each fold, the encoding is computed on the other four folds and applied to the held-out fold. This produces a leak-free OOF encoding for training. For the test set, the encoding is fitted once on all training data.

The function returns one encoded probability per class (three columns per original column), plus a derived spread feature measuring the gap between high-risk and low-risk probability.

In [ ]:
def kfold_te_multiclass(train_df, test_df, col, target_col,
                         n_classes=3, n_splits=5, smoothing=10, seed=SEED):
    """
    Out-of-fold multiclass target encoding with Bayesian smoothing.
    Returns OOF encoding for train and full encoding for test.
    """
    skf    = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof    = np.zeros((len(train_df), n_classes))
    priors = (train_df[target_col]
              .value_counts(normalize=True)
              .reindex(range(n_classes))
              .values)
    priors_s = pd.Series(priors, index=range(n_classes))

    for tr_idx, val_idx in skf.split(train_df, train_df[target_col]):
        tr     = train_df.iloc[tr_idx]
        counts = (tr.groupby(col)[target_col]
                    .value_counts()
                    .unstack(fill_value=0)
                    .reindex(columns=range(n_classes), fill_value=0))
        totals = counts.sum(axis=1)
        enc    = counts.add(smoothing * priors, axis=1).div(totals + smoothing, axis=0)

        val_vals  = train_df.iloc[val_idx][col]
        oof[val_idx] = enc.reindex(val_vals).fillna(priors_s).values

    # Full-train encoding for test
    counts_full = (train_df.groupby(col)[target_col]
                   .value_counts()
                   .unstack(fill_value=0)
                   .reindex(columns=range(n_classes), fill_value=0))
    totals_full = counts_full.sum(axis=1)
    enc_full    = counts_full.add(smoothing * priors, axis=1).div(totals_full + smoothing, axis=0)
    e_test      = enc_full.reindex(test_df[col]).fillna(priors_s)

    return oof, e_test.values

In [ ]:
# ── Target encode note_id ─────────────────────────────────────────────────────
oof_te, test_te = kfold_te_multiclass(train, test, 'note_id', TARGET, smoothing=10)

for i in range(3):
    train[f'note_te_p{i}'] = oof_te[:, i]
    test[f'note_te_p{i}']  = test_te[:, i]

print('note_id target encoding complete.')

In [ ]:
# ── Target encode situation_id ────────────────────────────────────────────────
oof_te_sit, test_te_sit = kfold_te_multiclass(train, test, 'situation_id', TARGET, smoothing=15)

for i in range(3):
    train[f'sit_te_p{i}'] = oof_te_sit[:, i]
    test[f'sit_te_p{i}']  = test_te_sit[:, i]

print('situation_id target encoding complete.')

In [ ]:
# ── Derived spread feature (high-risk probability minus low-risk probability) ─
for df in [train, test]:
    df['note_risk_spread'] = df['note_te_p2'] - df['note_te_p0']

---
## 8. Assemble the Final Feature List

The design matrix is assembled by dropping the identifiers, the raw target, and the original un-encoded categorical strings. All remaining columns are used as features. The categorical feature indices are recorded for models that use native categorical handling (CatBoost, LightGBM).

In [ ]:
FEATURES = [
    c for c in train.columns
    if c not in ['student_id', TARGET, 'counsellor_note'] + cat_cols
]

CAT_FEATURES = [
    'branch_enc', 'gender_enc', 'hostel_status_enc',
    'family_income_enc', 'parent_education_enc',
    'note_id', 'situation_id'
]

X      = train[FEATURES].copy()
y      = train[TARGET].copy()
X_test = test[FEATURES].copy()

cat_idx = [X.columns.get_loc(c) for c in CAT_FEATURES]

print(f'Total features : {len(FEATURES)}')
print(f'Train rows     : {X.shape[0]}')
print(f'Test rows      : {X_test.shape[0]}')

---
## 9. GBM Ensemble: 5-Fold Cross-Validated Training

Four gradient-boosted or tree-based models are trained using 5-fold stratified cross-validation. Each model outputs class probabilities, and the out-of-fold (OOF) probabilities are stored for later blending.

| Model | Key Strengths |
|-------|---------------|
| LightGBM | Fast, native categorical support, leaf-wise growth |
| XGBoost | Robust regularization, strong baseline |
| CatBoost | Ordered boosting, excellent on categorical features |
| ExtraTrees | High variance diversity via extra random splits |

All models use balanced class weights to compensate for the unequal class distribution visible in the confusion matrix. Early stopping with 100 rounds prevents overfitting.

In [ ]:
N_SPLITS = 5
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ── OOF and test probability arrays ──────────────────────────────────────────
oof_lgb  = np.zeros((len(X), 3));  pred_lgb = np.zeros((len(X_test), 3))
oof_xgb  = np.zeros((len(X), 3));  pred_xgb = np.zeros((len(X_test), 3))
oof_cb   = np.zeros((len(X), 3));  pred_cb  = np.zeros((len(X_test), 3))
oof_et   = np.zeros((len(X), 3));  pred_et  = np.zeros((len(X_test), 3))

In [ ]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    sw_tr = compute_sample_weight('balanced', y_tr)
    cw    = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_tr)

    # ── LightGBM ─────────────────────────────────────────────────────────────
    lgb_model = lgb.LGBMClassifier(
        objective        = 'multiclass',
        num_class        = 3,
        n_estimators     = 2000,
        learning_rate    = 0.03,
        num_leaves       = 31,
        feature_fraction = 0.8,
        bagging_fraction = 0.8,
        bagging_freq     = 1,
        reg_alpha        = 0.1,
        reg_lambda       = 0.1,
        random_state     = SEED,
        verbose          = -1
    )
    lgb_model.fit(
        X_tr, y_tr,
        sample_weight      = sw_tr,
        eval_set           = [(X_val, y_val)],
        eval_metric        = 'multi_logloss',
        categorical_feature= CAT_FEATURES,
        callbacks          = [lgb.early_stopping(100, verbose=False)]
    )
    oof_lgb[val_idx]  = lgb_model.predict_proba(X_val)
    pred_lgb          += lgb_model.predict_proba(X_test) / N_SPLITS

    # ── XGBoost ──────────────────────────────────────────────────────────────
    xgb_model = xgb.XGBClassifier(
        objective            = 'multi:softprob',
        num_class            = 3,
        n_estimators         = 2000,
        learning_rate        = 0.03,
        max_depth            = 6,
        subsample            = 0.8,
        colsample_bytree     = 0.8,
        reg_alpha            = 0.1,
        reg_lambda           = 1.0,
        random_state         = SEED,
        eval_metric          = 'mlogloss',
        early_stopping_rounds= 100
    )
    xgb_model.fit(X_tr, y_tr, sample_weight=sw_tr,
                  eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = xgb_model.predict_proba(X_val)
    pred_xgb          += xgb_model.predict_proba(X_test) / N_SPLITS

    # ── CatBoost ─────────────────────────────────────────────────────────────
    cb_model = CatBoostClassifier(
        loss_function         = 'MultiClass',
        iterations            = 2000,
        learning_rate         = 0.03,
        depth                 = 6,
        l2_leaf_reg           = 3,
        class_weights         = cw.tolist(),
        cat_features          = cat_idx,
        random_seed           = SEED,
        verbose               = False,
        early_stopping_rounds = 100
    )
    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cb[val_idx]   = cb_model.predict_proba(X_val)
    pred_cb           += cb_model.predict_proba(X_test) / N_SPLITS

    # ── ExtraTrees ───────────────────────────────────────────────────────────
    et_model = ExtraTreesClassifier(
        n_estimators     = 600,
        min_samples_leaf = 3,
        class_weight     = 'balanced',
        n_jobs           = -1,
        random_state     = SEED
    )
    et_model.fit(X_tr, y_tr)
    oof_et[val_idx]   = et_model.predict_proba(X_val)
    pred_et           += et_model.predict_proba(X_test) / N_SPLITS

    # ── Fold summary ─────────────────────────────────────────────────────────
    fold_preds = np.argmax(
        (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cb[val_idx] + oof_et[val_idx]) / 4,
        axis=1
    )
    fold_f1 = f1_score(y_val, fold_preds, average='macro')
    print(f'Fold {fold + 1}  GBM ensemble macro-F1: {fold_f1:.4f}')

In [ ]:
# ── Average the four models into the GBM ensemble ────────────────────────────
oof_ens4  = (oof_lgb + oof_xgb + oof_cb + oof_et) / 4
test_ens4 = (pred_lgb + pred_xgb + pred_cb + pred_et) / 4

gbm_oof_f1 = f1_score(y, np.argmax(oof_ens4, axis=1), average='macro')
print(f'GBM 4-model OOF macro-F1: {gbm_oof_f1:.4f}')

---
## 10. Multimodal Deep Learning

A single neural network processes three different input modalities simultaneously.

- **Tabular MLP** — scaled numeric features and learned categorical embeddings are passed through a two-layer MLP with batch normalisation and dropout
- **Attendance BiLSTM** — the attendance series is shaped as a (24 weeks × 3 subjects) tensor and processed by a bidirectional LSTM, capturing temporal trends in attendance
- **Notes BiLSTM** — the counsellor note is tokenised and encoded by a small embedding layer followed by a bidirectional LSTM, capturing sequential language patterns

The three branch outputs are concatenated and passed through a final classification head.

### 10a. Attendance Sequence Tensor

The long-format attendance table is pivoted into a (students × 24 timesteps × 3 subjects) tensor. Missing values are imputed with the student's own row mean and then with the global mean.

In [ ]:
# ── Combine train and test student IDs for joint sequence construction ────────
all_students = pd.concat([train['student_id'], test['student_id']]).reset_index(drop=True)
n_train      = len(train)

subjects  = sorted(att['subject'].unique())
full_cols = pd.MultiIndex.from_product([range(1, 25), subjects], names=['t', 'subject'])

piv      = att.pivot_table(index='student_id', columns=['t', 'subject'], values='attendance_pct')
piv      = piv.reindex(index=all_students, columns=full_cols)
row_mean = piv.mean(axis=1)
piv      = piv.apply(lambda col: col.fillna(row_mean))
piv      = piv.fillna(piv.values.mean())

ATT_SEQ        = piv.values.reshape(len(all_students), 24, len(subjects)).astype(np.float32)
ATT_SEQ_train  = ATT_SEQ[:n_train]
ATT_SEQ_test   = ATT_SEQ[n_train:]

print(f'Attendance sequence tensor shape (train): {ATT_SEQ_train.shape}')

### 10b. Counsellor Note Text Sequences

Each note is lowercased, punctuation-stripped, and tokenised into words. A small vocabulary is built from the training corpus, each word receives an integer index, and notes are zero-padded or truncated to a fixed length of 10 tokens.

In [ ]:
notes_s   = notes.set_index('student_id').reindex(all_students)['counsellor_note']
tok_series = (
    notes_s
    .str.lower()
    .str.replace('.', ' ', regex=False)
    .str.replace(',', ' ', regex=False)
    .str.split()
)

vocab     = sorted(set(w for toks in tok_series for w in toks))
word2idx  = {w: i + 1 for i, w in enumerate(vocab)}
MAX_LEN   = 10
VOCAB_SIZE = len(vocab)

def tok_to_seq(toks):
    seq = [word2idx[w] for w in toks][:MAX_LEN]
    return seq + [0] * (MAX_LEN - len(seq))

TEXT_SEQ       = np.array([tok_to_seq(t) for t in tok_series], dtype=np.int64)
TEXT_SEQ_train = TEXT_SEQ[:n_train]
TEXT_SEQ_test  = TEXT_SEQ[n_train:]

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Text sequence shape (train): {TEXT_SEQ_train.shape}')

### 10c. Numeric and Categorical Tabular Inputs

In [ ]:
EMBED_CATS   = CAT_FEATURES
NUMERIC_FEATS = [c for c in FEATURES if c not in EMBED_CATS]

scaler     = StandardScaler()
NUM_train  = scaler.fit_transform(X[NUMERIC_FEATS].values.astype(np.float32))
NUM_test   = scaler.transform(X_test[NUMERIC_FEATS].values.astype(np.float32))

CAT_train        = X[EMBED_CATS].values.astype(np.int64)
CAT_test         = X_test[EMBED_CATS].values.astype(np.int64)
CAT_CARDINALITIES = [int(X[c].max()) + 2 for c in EMBED_CATS]

y_arr = y.values.astype(np.int64)

print(f'Numeric features : {NUM_train.shape[1]}')
print(f'Embed categories : {len(EMBED_CATS)}')

### 10d. Dataset Class and Model Architecture

In [ ]:
class StudentDataset(Dataset):
    """
    PyTorch Dataset wrapping four input modalities and an optional label.
    """
    def __init__(self, num_f, cat_f, att_seq, text_seq, labels=None):
        self.num_f    = torch.tensor(num_f,    dtype=torch.float32)
        self.cat_f    = torch.tensor(cat_f,    dtype=torch.long)
        self.att_seq  = torch.tensor(att_seq,  dtype=torch.float32)
        self.text_seq = torch.tensor(text_seq, dtype=torch.long)
        self.labels   = torch.tensor(labels,   dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.num_f)

    def __getitem__(self, i):
        if self.labels is not None:
            return self.num_f[i], self.cat_f[i], self.att_seq[i], self.text_seq[i], self.labels[i]
        return self.num_f[i], self.cat_f[i], self.att_seq[i], self.text_seq[i]

In [ ]:
class MultiModalNet(nn.Module):
    """
    Three-branch multimodal network.

    Branch 1  Tabular MLP: numeric features + categorical embeddings
    Branch 2  Attendance BiLSTM: weekly attendance sequence per subject
    Branch 3  Notes BiLSTM: tokenised counsellor note
    """
    def __init__(self, n_numeric, cat_cardinalities, vocab_size, n_subjects=3, n_classes=3):
        super().__init__()

        # ── Categorical embeddings ────────────────────────────────────────────
        emb_dims         = [min(24, (c + 1) // 2) for c in cat_cardinalities]
        self.embeddings  = nn.ModuleList([
            nn.Embedding(c, d) for c, d in zip(cat_cardinalities, emb_dims)
        ])
        total_emb = sum(emb_dims)

        # ── Tabular MLP ───────────────────────────────────────────────────────
        self.tab_mlp = nn.Sequential(
            nn.Linear(n_numeric + total_emb, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )

        # ── Attendance BiLSTM ─────────────────────────────────────────────────
        self.att_lstm = nn.LSTM(
            input_size  = n_subjects,
            hidden_size = 32,
            num_layers  = 1,
            batch_first = True,
            bidirectional = True
        )

        # ── Notes BiLSTM ──────────────────────────────────────────────────────
        self.text_emb  = nn.Embedding(vocab_size + 1, 16, padding_idx=0)
        self.text_lstm = nn.LSTM(
            input_size  = 16,
            hidden_size = 16,
            num_layers  = 1,
            batch_first = True,
            bidirectional = True
        )

        # ── Classification head ───────────────────────────────────────────────
        combined_dim = 64 + 32 * 2 + 16 * 2
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x_num, x_cat, x_att, x_text):
        # Tabular branch
        embs = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        tab  = self.tab_mlp(torch.cat([x_num, embs], dim=1))

        # Attendance branch
        _, (h_att, _) = self.att_lstm(x_att)
        att_out = torch.cat([h_att[0], h_att[1]], dim=1)

        # Notes branch
        e = self.text_emb(x_text)
        _, (h_txt, _) = self.text_lstm(e)
        txt_out = torch.cat([h_txt[0], h_txt[1]], dim=1)

        return self.head(torch.cat([tab, att_out, txt_out], dim=1))

### 10e. 5-Fold Training with Early Stopping

The neural network is trained in the same 5-fold scheme as the GBM models so the OOF probabilities align row for row. Training uses the AdamW optimiser with weight decay, a ReduceLROnPlateau scheduler monitoring macro-F1, and early stopping with patience of 8 epochs.

In [ ]:
oof_nn   = np.zeros((len(X), 3))
pred_nn  = np.zeros((len(X_test), 3))

EPOCHS     = 60
PATIENCE   = 8
BATCH_SIZE = 128

for fold, (tr_idx, val_idx) in enumerate(skf.split(NUM_train, y_arr)):

    # ── Build DataLoaders ────────────────────────────────────────────────────
    train_ds = StudentDataset(
        NUM_train[tr_idx], CAT_train[tr_idx],
        ATT_SEQ_train[tr_idx], TEXT_SEQ_train[tr_idx], y_arr[tr_idx]
    )
    val_ds = StudentDataset(
        NUM_train[val_idx], CAT_train[val_idx],
        ATT_SEQ_train[val_idx], TEXT_SEQ_train[val_idx], y_arr[val_idx]
    )
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=512,         shuffle=False)

    # ── Instantiate model, loss, optimiser, scheduler ────────────────────────
    model     = MultiModalNet(NUM_train.shape[1], CAT_CARDINALITIES, VOCAB_SIZE).to(device)
    cw        = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_arr[tr_idx])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_f1, best_state, patience_ctr = -1, None, 0

    # ── Epoch loop ───────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):
        model.train()
        for num_f, cat_f, att_f, txt_f, lbl in train_dl:
            num_f, cat_f, att_f, txt_f, lbl = [t.to(device) for t in (num_f, cat_f, att_f, txt_f, lbl)]
            optimizer.zero_grad()
            loss = criterion(model(num_f, cat_f, att_f, txt_f), lbl)
            loss.backward()
            optimizer.step()

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for num_f, cat_f, att_f, txt_f, lbl in val_dl:
                num_f, cat_f, att_f, txt_f = [t.to(device) for t in (num_f, cat_f, att_f, txt_f)]
                out = model(num_f, cat_f, att_f, txt_f)
                val_preds.append(out.argmax(1).cpu().numpy())
                val_true.append(lbl.numpy())

        val_preds = np.concatenate(val_preds)
        val_true  = np.concatenate(val_true)
        val_f1    = f1_score(val_true, val_preds, average='macro')
        scheduler.step(val_f1)

        if val_f1 > best_f1:
            best_f1      = val_f1
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    print(f'Fold {fold + 1}  NN best val macro-F1: {best_f1:.4f}')

    # ── Collect OOF and test probabilities ───────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        val_ds_full = StudentDataset(
            NUM_train[val_idx], CAT_train[val_idx],
            ATT_SEQ_train[val_idx], TEXT_SEQ_train[val_idx]
        )
        val_dl_full = DataLoader(val_ds_full, batch_size=512, shuffle=False)
        probs = []
        for num_f, cat_f, att_f, txt_f in val_dl_full:
            num_f, cat_f, att_f, txt_f = [t.to(device) for t in (num_f, cat_f, att_f, txt_f)]
            probs.append(torch.softmax(model(num_f, cat_f, att_f, txt_f), dim=1).cpu().numpy())
        oof_nn[val_idx] = np.concatenate(probs)

        test_ds = StudentDataset(NUM_test, CAT_test, ATT_SEQ_test, TEXT_SEQ_test)
        test_dl = DataLoader(test_ds, batch_size=512, shuffle=False)
        probs = []
        for num_f, cat_f, att_f, txt_f in test_dl:
            num_f, cat_f, att_f, txt_f = [t.to(device) for t in (num_f, cat_f, att_f, txt_f)]
            probs.append(torch.softmax(model(num_f, cat_f, att_f, txt_f), dim=1).cpu().numpy())
        pred_nn += np.concatenate(probs) / N_SPLITS

nn_oof_f1 = f1_score(y, np.argmax(oof_nn, axis=1), average='macro')
print(f'\nNeural Network OOF macro-F1: {nn_oof_f1:.4f}')

---
## 11. Blend, Tune, Evaluate, and Save Submission

The GBM ensemble and the neural network are combined with a scalar blending weight beta, where beta = 1 means pure GBM and beta = 0 means pure neural network. On top of the blend, three per-class multiplicative weights (w0, w1, w2) are applied before taking the argmax. This two-level tuning is done by grid search on the OOF predictions, so no test-set information leaks into the optimisation.

After finding the best weights, the final predictions are decoded, evaluated, and written to `submission_final.csv`.

In [ ]:
# ── Grid search over blend weight and per-class weights ──────────────────────
best_score = -1
best_beta  = 0.5
best_w     = (1.0, 1.0, 1.0)

for beta in np.arange(0.0, 1.01, 0.1):
    blend = beta * oof_ens4 + (1 - beta) * oof_nn
    for w1 in np.arange(0.4, 1.81, 0.02):
        for w2 in np.arange(0.3, 1.61, 0.02):
            w     = np.array([1.0, w1, w2])
            score = f1_score(y, np.argmax(blend * w, axis=1), average='macro')
            if score > best_score:
                best_score = score
                best_beta  = beta
                best_w     = w

print(f'Optimal blend weight (GBM share)  : {best_beta:.2f}')
print(f'Optimal per-class weights          : {best_w}')
print(f'Tuned OOF macro-F1                 : {best_score:.4f}')

In [ ]:
# ── Assemble final OOF and test predictions ───────────────────────────────────
final_oof  = best_beta * oof_ens4  + (1 - best_beta) * oof_nn
final_test = best_beta * test_ens4 + (1 - best_beta) * pred_nn

oof_preds_final  = np.argmax(final_oof  * best_w, axis=1)
test_preds_final = np.argmax(final_test * best_w, axis=1)

In [ ]:
# ── Per-class classification report ──────────────────────────────────────────
print(classification_report(y, oof_preds_final, target_names=['Low', 'Medium', 'High']))

---
## 12. Confusion Matrix

The confusion matrix shows how predictions are distributed across the three classes. Each row represents the true label and each column the predicted label. The diagonal entries are correct classifications; off-diagonal entries are misclassifications.

Key observations from the final result:
- **Low risk** is predicted very well at 82.6% recall, with most errors being confused with Medium
- **Medium risk** is the hardest class at 58.1% recall, with leakage in both directions
- **High risk** achieves 77.1% recall, with confusions predominantly toward Medium

In [ ]:
cm = confusion_matrix(y, oof_preds_final)
labels = ['Low', 'Medium', 'High']

fig, ax = plt.subplots(figsize=(7, 5.5))

# Annotate with both counts and row percentages
cm_pct   = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot    = np.array([[f'{cm[i, j]}\n({cm_pct[i, j]:.1f}%)' for j in range(3)] for i in range(3)])

sns.heatmap(
    cm,
    annot    = annot,
    fmt      = '',
    cmap     = 'Blues',
    xticklabels = labels,
    yticklabels = labels,
    linewidths  = 0.5,
    linecolor   = 'white',
    ax          = ax
)

ax.set_xlabel('Predicted Label (Submission)', fontsize=12)
ax.set_ylabel('True Label (Solution)',         fontsize=12)
ax.set_title(
    f'Final Model Evaluation\nAccuracy: {(cm.diagonal().sum() / cm.sum() * 100):.2f}%  |  Macro-F1: {best_score:.4f}',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('confusion_matrix_final.png', dpi=150)
plt.show()

---
## 13. Confidence Confusion Matrix

The standard confusion matrix only shows hard predictions. The confidence confusion matrix shows the **average predicted probability for each true-label row**, giving a richer view of how certain the model is when it predicts each class.

Each cell (i, j) is the mean softmax probability the model assigned to class j for samples whose true label is i. The diagonal cells show how confident the model is when it is correct. Off-diagonal cells reveal where it hedges or expresses misplaced confidence.

In [ ]:
# ── Compute mean predicted probability per true class ─────────────────────────
conf_matrix = np.zeros((3, 3))

for true_class in range(3):
    mask = (y_arr == true_class)
    conf_matrix[true_class] = final_oof[mask].mean(axis=0)

labels = ['Low', 'Medium', 'High']

fig, ax = plt.subplots(figsize=(7, 5.5))

annot_conf = np.array([[f'{conf_matrix[i, j]:.3f}' for j in range(3)] for i in range(3)])

sns.heatmap(
    conf_matrix,
    annot    = annot_conf,
    fmt      = '',
    cmap     = 'YlOrRd',
    vmin     = 0.0,
    vmax     = 1.0,
    xticklabels = labels,
    yticklabels = labels,
    linewidths  = 0.5,
    linecolor   = 'white',
    ax          = ax
)

ax.set_xlabel('Predicted Class (Probability)',  fontsize=12)
ax.set_ylabel('True Label (Solution)',           fontsize=12)
ax.set_title(
    'Confidence Confusion Matrix\nMean Predicted Probability per True Class',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('confidence_confusion_matrix.png', dpi=150)
plt.show()

print('\nConfidence matrix (rows = true class, columns = predicted class):')
print(pd.DataFrame(conf_matrix, index=labels, columns=labels).round(4))

---
## 14. Save Submission File

The test predictions are saved as `submission_final.csv` with two columns: `student_id` and `dropout_risk`. The distribution of predicted classes in the submission is printed as a sanity check.

In [ ]:
submission = pd.DataFrame({
    'student_id'   : test['student_id'],
    'dropout_risk' : test_preds_final
})

submission.to_csv('submission_final.csv', index=False)

print('Submission saved to submission_final.csv')
print('\nPredicted class distribution in submission:')
print(submission['dropout_risk'].value_counts(normalize=True).round(4))
submission.head()